# 06. Bias zespo?owy i modele hierarchiczne


Notebook zbiera analizy dotycz?ce wp?ywu zespo?u i samochodu na klasyfikacj? kierowcy.
Obejmuje predykcj? zespo?u, testy kierowc?w w tym samym zespole, wariant hierarchiczny oraz model team-aware.
Kod pochodzi ze skrypt?w `06_team_bias_and_team_signature_analysis.py`, `07_hierarchical_team_driver_model.py` i `08_team_aware_driver_models.py`.

**Uwaga:** kom?rki z kodem zawieraj? ?r?d?owe skrypty, ale wywo?anie `main()` jest celowo zakomentowane. Dzi?ki temu przypadkowe `Run All` nie uruchomi d?ugich pobra? danych ani trening?w. Aby wykona? dan? sekcj?, najlepiej uruchomi? j? ?wiadomie albo u?y? `%run nazwa_skryptu.py`.


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
EXPORT_DIR = ROOT / 'exports'
FIG_DIR = ROOT / 'figures' / 'thesis'

print(f'Root: {ROOT}')
print(f'Exports: {EXPORT_DIR}')
print(f'Figures: {FIG_DIR}')


## Predykcja zespo?u, testy w tym samym zespole i wa?no?? cech

?r?d?o: `06_team_bias_and_team_signature_analysis.py`

Je?eli chcesz uruchomi? oryginalny skrypt bez wchodzenia w definicje funkcji, u?yj:

```python
%run 06_team_bias_and_team_signature_analysis.py
```


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler


EXPORT_DIR = Path("exports")
RANDOM_STATE = 42
N_SPLITS = 5


DATASETS = {
    "short_qualifying_2023_2025": EXPORT_DIR / "lap_features_model_min_40_laps.csv",
    "long_qualifying_2018_2025_top5": EXPORT_DIR / "balanced_top5_lap_features_2018_2025_strict.csv",
    "race_2025_clean_laps_top5": EXPORT_DIR / "race_2025_balanced_top5_lap_features.csv",
}


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def session_column(df: pd.DataFrame) -> str:
    if "session_key" in df.columns:
        return "session_key"
    return "round"


def feature_sets(df: pd.DataFrame) -> dict[str, list[str]]:
    identifiers = {
        "lap_key",
        "Driver",
        "Team",
        "season",
        "round",
        "LapNumber",
        "event_name",
        "EventName",
        "session_name",
        "session_key",
        "TrackStatus",
        "lap_time_not_null",
        "dry_session",
        "accurate",
        "not_deleted",
        "not_fastf1_generated",
        "track_status_green",
        "not_pit_in_out",
        "within_2s_of_stint_median",
        "within_3s_of_stint_median",
    }
    weak_or_constant = {
        "throttle_min",
        "brake_min",
        "brake_median",
        "brake_hard_frac",
        "drs_toggle_count",
        "brake_diff_mean",
    }
    time_features = {
        "LapTimeSeconds",
        "Sector1TimeSeconds",
        "Sector2TimeSeconds",
        "Sector3TimeSeconds",
        "sector_sum_seconds",
        "sector1_share",
        "sector2_share",
        "sector3_share",
        "stint_median_lap_seconds",
        "lap_time_delta_to_stint_median",
    }
    race_context = {
        "Compound",
        "TyreLife",
        "Stint",
        "SessionPart",
    }
    drs_features = {
        "DRS",
        "drs_active_frac",
    }

    all_missing = {c for c in df.columns if df[c].isna().all()}

    def keep(excluded: set[str]) -> list[str]:
        cols = [c for c in df.columns if c not in excluded and c not in all_missing]
        return [c for c in cols if c in df.columns]

    return {
        "all_telemetry_features": keep(identifiers | weak_or_constant),
        "no_time_features": keep(identifiers | weak_or_constant | time_features),
        "style_core_no_time_context": keep(
            identifiers | weak_or_constant | time_features | race_context | drs_features
        ),
    }


def build_logreg(numeric_features: list[str], categorical_features: list[str]) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_features,
            ),
        ]
    )
    model = LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=2000,
        random_state=RANDOM_STATE,
    )
    return Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])


def run_grouped_logreg(
    df: pd.DataFrame,
    dataset_name: str,
    task_name: str,
    target_col: str,
    feature_config: str,
    feature_cols: list[str],
) -> dict[str, object]:
    df = df.dropna(subset=[target_col]).copy()
    label_counts = df[target_col].value_counts()
    if len(label_counts) < 2:
        raise ValueError(f"{dataset_name}/{task_name}: less than two target classes")

    group_col = session_column(df)
    n_splits = min(N_SPLITS, int(label_counts.min()))
    if n_splits < 2:
        raise ValueError(f"{dataset_name}/{task_name}: not enough samples per class")

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df[target_col].astype(str))
    groups = df[group_col].astype(str).to_numpy()
    X = df[feature_cols].copy()

    categorical_features = [c for c in feature_cols if X[c].dtype == "object"]
    numeric_features = [c for c in feature_cols if c not in categorical_features]

    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_pred = np.full(len(df), -1, dtype=int)
    train_scores = []
    test_scores = []

    for train_idx, test_idx in cv.split(X, y, groups):
        model = build_logreg(numeric_features, categorical_features)
        model.fit(X.iloc[train_idx], y[train_idx])
        train_pred = model.predict(X.iloc[train_idx])
        test_pred = model.predict(X.iloc[test_idx])
        oof_pred[test_idx] = test_pred
        train_scores.append(accuracy_score(y[train_idx], train_pred))
        test_scores.append(accuracy_score(y[test_idx], test_pred))

    return {
        "dataset": dataset_name,
        "task": task_name,
        "target": target_col,
        "feature_config": feature_config,
        "n_samples": len(df),
        "n_classes": len(label_counts),
        "n_groups": df[group_col].nunique(),
        "n_features": len(feature_cols),
        "min_class_count": int(label_counts.min()),
        "max_class_count": int(label_counts.max()),
        "oof_accuracy": accuracy_score(y, oof_pred),
        "oof_balanced_accuracy": balanced_accuracy_score(y, oof_pred),
        "oof_macro_f1": f1_score(y, oof_pred, average="macro"),
        "mean_train_accuracy": float(np.mean(train_scores)),
        "mean_test_accuracy": float(np.mean(test_scores)),
        "mean_accuracy_gap": float(np.mean(train_scores) - np.mean(test_scores)),
    }


def fit_logreg_feature_importance(
    df: pd.DataFrame,
    dataset_name: str,
    task_name: str,
    target_col: str,
    feature_config: str,
    feature_cols: list[str],
    top_n: int = 30,
) -> pd.DataFrame:
    df = df.dropna(subset=[target_col]).copy()
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df[target_col].astype(str))
    X = df[feature_cols].copy()

    categorical_features = [c for c in feature_cols if X[c].dtype == "object"]
    numeric_features = [c for c in feature_cols if c not in categorical_features]

    model = build_logreg(numeric_features, categorical_features)
    model.fit(X, y)

    feature_names = model.named_steps["preprocessor"].get_feature_names_out()
    coefs = model.named_steps["model"].coef_
    classes = label_encoder.classes_

    rows = []
    if len(classes) == 2 and coefs.shape[0] == 1:
        coef_matrix = np.vstack([-coefs[0], coefs[0]])
    else:
        coef_matrix = coefs

    for class_name, class_coefs in zip(classes, coef_matrix):
        order = np.argsort(np.abs(class_coefs))[::-1][:top_n]
        for rank, idx in enumerate(order, start=1):
            rows.append(
                {
                    "dataset": dataset_name,
                    "task": task_name,
                    "target": target_col,
                    "feature_config": feature_config,
                    "class_name": class_name,
                    "rank": rank,
                    "feature": feature_names[idx],
                    "coef": class_coefs[idx],
                    "abs_coef": abs(class_coefs[idx]),
                }
            )
    return pd.DataFrame(rows)


def load_datasets() -> dict[str, pd.DataFrame]:
    loaded = {}
    for name, path in DATASETS.items():
        if path.exists():
            loaded[name] = pd.read_csv(path)
    return loaded


def team_coverage_rows(datasets: dict[str, pd.DataFrame]) -> list[dict[str, object]]:
    rows = []
    for dataset_name, df in datasets.items():
        if "Driver" not in df.columns or "Team" not in df.columns:
            continue
        counts = (
            df.groupby(["Driver", "Team"], dropna=False)
            .size()
            .reset_index(name="n_laps")
            .sort_values(["Driver", "n_laps"], ascending=[True, False])
        )
        for row in counts.to_dict("records"):
            row["dataset"] = dataset_name
            rows.append(row)
    return rows


def team_classification(datasets: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for dataset_name, df in datasets.items():
        if "Team" not in df.columns:
            continue
        team_counts = df["Team"].value_counts()
        eligible_teams = team_counts[team_counts >= 30].index
        filtered = df[df["Team"].isin(eligible_teams)].copy()
        for feature_config, cols in feature_sets(filtered).items():
            rows.append(
                run_grouped_logreg(
                    filtered,
                    dataset_name,
                    "predict_team",
                    "Team",
                    feature_config,
                    cols,
                )
            )
    return pd.DataFrame(rows)


def team_feature_importance(datasets: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for dataset_name, df in datasets.items():
        if "Team" not in df.columns:
            continue
        team_counts = df["Team"].value_counts()
        eligible_teams = team_counts[team_counts >= 30].index
        filtered = df[df["Team"].isin(eligible_teams)].copy()
        for feature_config, cols in feature_sets(filtered).items():
            rows.append(
                fit_logreg_feature_importance(
                    filtered,
                    dataset_name,
                    "predict_team",
                    "Team",
                    feature_config,
                    cols,
                )
            )
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def same_team_driver_tests(datasets: dict[str, pd.DataFrame]) -> pd.DataFrame:
    tests = [
        {
            "dataset": "race_2025_clean_laps_top5",
            "team": "Ferrari",
            "drivers": ["HAM", "LEC"],
            "label": "same_team_ferrari_ham_vs_lec_2025_race",
        },
        {
            "dataset": "short_qualifying_2023_2025",
            "team": "Ferrari",
            "drivers": ["LEC", "SAI"],
            "label": "same_team_ferrari_lec_vs_sai_2023_2024_qual",
        },
        {
            "dataset": "long_qualifying_2018_2025_top5",
            "team": "Ferrari",
            "drivers": ["LEC", "SAI", "HAM"],
            "label": "same_team_ferrari_lec_sai_ham_long_qual",
        },
    ]
    rows = []
    for test in tests:
        df = datasets.get(test["dataset"])
        if df is None:
            continue
        filtered = df[
            (df["Team"] == test["team"]) & (df["Driver"].isin(test["drivers"]))
        ].copy()
        if filtered["Driver"].nunique() < 2:
            continue
        for feature_config, cols in feature_sets(filtered).items():
            rows.append(
                run_grouped_logreg(
                    filtered,
                    test["dataset"],
                    test["label"],
                    "Driver",
                    feature_config,
                    cols,
                )
            )
    return pd.DataFrame(rows)


def same_team_feature_importance(datasets: dict[str, pd.DataFrame]) -> pd.DataFrame:
    tests = [
        {
            "dataset": "race_2025_clean_laps_top5",
            "team": "Ferrari",
            "drivers": ["HAM", "LEC"],
            "label": "same_team_ferrari_ham_vs_lec_2025_race",
        },
        {
            "dataset": "short_qualifying_2023_2025",
            "team": "Ferrari",
            "drivers": ["LEC", "SAI"],
            "label": "same_team_ferrari_lec_vs_sai_2023_2024_qual",
        },
        {
            "dataset": "long_qualifying_2018_2025_top5",
            "team": "Ferrari",
            "drivers": ["LEC", "SAI", "HAM"],
            "label": "same_team_ferrari_lec_sai_ham_long_qual",
        },
    ]
    rows = []
    for test in tests:
        df = datasets.get(test["dataset"])
        if df is None:
            continue
        filtered = df[
            (df["Team"] == test["team"]) & (df["Driver"].isin(test["drivers"]))
        ].copy()
        if filtered["Driver"].nunique() < 2:
            continue
        for feature_config, cols in feature_sets(filtered).items():
            rows.append(
                fit_logreg_feature_importance(
                    filtered,
                    test["dataset"],
                    test["label"],
                    "Driver",
                    feature_config,
                    cols,
                )
            )
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def group_feature_means(datasets: dict[str, pd.DataFrame]) -> pd.DataFrame:
    core_features = [
        "speed_mean",
        "speed_max",
        "speed_q90",
        "speed_std",
        "speed_min",
        "throttle_mean",
        "throttle_std",
        "throttle_zero_frac",
        "throttle_mid_frac",
        "throttle_full_frac",
        "throttle_diff_std",
        "throttle_diff_abs_mean",
        "brake_active_frac",
        "brake_on_count",
        "brake_std",
        "gear_mean",
        "gear_std",
        "gear_change_count",
        "rpm_mean",
        "rpm_std",
        "rpm_max",
        "rpm_diff_std",
    ]
    rows = []
    for dataset_name, df in datasets.items():
        available = [c for c in core_features if c in df.columns]
        if "Team" in df.columns:
            team_means = df.groupby("Team", dropna=False)[available].mean().reset_index()
            team_means.insert(0, "group_type", "Team")
            team_means.insert(0, "dataset", dataset_name)
            team_means = team_means.rename(columns={"Team": "group_name"})
            rows.append(team_means)
        if "Driver" in df.columns:
            driver_means = df.groupby("Driver", dropna=False)[available].mean().reset_index()
            driver_means.insert(0, "group_type", "Driver")
            driver_means.insert(0, "dataset", dataset_name)
            driver_means = driver_means.rename(columns={"Driver": "group_name"})
            rows.append(driver_means)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def main() -> None:
    datasets = load_datasets()

    coverage = pd.DataFrame(team_coverage_rows(datasets))
    team_summary = team_classification(datasets)
    same_team_summary = same_team_driver_tests(datasets)
    team_importance = team_feature_importance(datasets)
    same_team_importance = same_team_feature_importance(datasets)
    feature_means = group_feature_means(datasets)

    team_summary = team_summary.sort_values(
        ["dataset", "oof_macro_f1", "oof_accuracy"], ascending=[True, False, False]
    )
    same_team_summary = same_team_summary.sort_values(
        ["task", "oof_macro_f1", "oof_accuracy"], ascending=[True, False, False]
    )

    export_csv(coverage, "team_bias_driver_team_coverage")
    export_csv(team_summary, "team_bias_team_classification_summary")
    export_csv(same_team_summary, "team_bias_same_team_driver_summary")
    export_csv(team_importance, "team_bias_team_feature_importance")
    export_csv(same_team_importance, "team_bias_same_team_driver_feature_importance")
    export_csv(feature_means, "team_bias_group_feature_means")

    print("Team-bias analysis complete.")
    print("\nTeam prediction:")
    print(team_summary.to_string(index=False))
    print("\nSame-team driver tests:")
    print(same_team_summary.to_string(index=False))


# Notebook safety: odkomentuj, je?eli chcesz uruchomi? t? sekcj?.
# main()


## Model hierarchiczny: najpierw zesp??, potem kierowca

?r?d?o: `07_hierarchical_team_driver_model.py`

Je?eli chcesz uruchomi? oryginalny skrypt bez wchodzenia w definicje funkcji, u?yj:

```python
%run 07_hierarchical_team_driver_model.py
```


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler


EXPORT_DIR = Path("exports")
RANDOM_STATE = 42
N_SPLITS = 5


DATASETS = {
    "short_qualifying_2023_2025_top5": {
        "path": EXPORT_DIR / "lap_features_model_min_40_laps.csv",
        "drivers": ["ALB", "SAI", "VER", "OCO", "LEC"],
    },
    "long_qualifying_2018_2025_top5": {
        "path": EXPORT_DIR / "balanced_top5_lap_features_2018_2025_strict.csv",
        "drivers": ["SAI", "VER", "LEC", "OCO", "HAM"],
    },
    "race_2025_clean_laps_top5": {
        "path": EXPORT_DIR / "race_2025_balanced_top5_lap_features.csv",
        "drivers": ["SAI", "VER", "LEC", "OCO", "HAM"],
    },
}


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def session_column(df: pd.DataFrame) -> str:
    if "session_key" in df.columns:
        return "session_key"
    return "round"


def style_feature_cols(df: pd.DataFrame) -> list[str]:
    excluded = {
        "lap_key",
        "Driver",
        "Team",
        "season",
        "round",
        "LapNumber",
        "event_name",
        "EventName",
        "session_name",
        "session_key",
        "TrackStatus",
        "lap_time_not_null",
        "dry_session",
        "accurate",
        "not_deleted",
        "not_fastf1_generated",
        "track_status_green",
        "not_pit_in_out",
        "within_2s_of_stint_median",
        "within_3s_of_stint_median",
        "LapTimeSeconds",
        "Sector1TimeSeconds",
        "Sector2TimeSeconds",
        "Sector3TimeSeconds",
        "sector_sum_seconds",
        "sector1_share",
        "sector2_share",
        "sector3_share",
        "stint_median_lap_seconds",
        "lap_time_delta_to_stint_median",
        "Compound",
        "TyreLife",
        "Stint",
        "SessionPart",
        "throttle_min",
        "brake_min",
        "brake_median",
        "brake_hard_frac",
        "drs_toggle_count",
        "brake_diff_mean",
        "drs_active_frac",
    }
    all_missing = {c for c in df.columns if df[c].isna().all()}
    return [c for c in df.columns if c not in excluded and c not in all_missing]


def feature_sets(df: pd.DataFrame) -> dict[str, list[str]]:
    base_excluded = {
        "lap_key",
        "Driver",
        "Team",
        "season",
        "round",
        "LapNumber",
        "event_name",
        "EventName",
        "session_name",
        "session_key",
        "TrackStatus",
        "lap_time_not_null",
        "dry_session",
        "accurate",
        "not_deleted",
        "not_fastf1_generated",
        "track_status_green",
        "not_pit_in_out",
        "within_2s_of_stint_median",
        "within_3s_of_stint_median",
    }
    weak_or_constant = {
        "throttle_min",
        "brake_min",
        "brake_median",
        "brake_hard_frac",
        "drs_toggle_count",
        "brake_diff_mean",
    }
    time_features = {
        "LapTimeSeconds",
        "Sector1TimeSeconds",
        "Sector2TimeSeconds",
        "Sector3TimeSeconds",
        "sector_sum_seconds",
        "sector1_share",
        "sector2_share",
        "sector3_share",
        "stint_median_lap_seconds",
        "lap_time_delta_to_stint_median",
    }
    context_features = {
        "Compound",
        "TyreLife",
        "Stint",
        "SessionPart",
        "drs_active_frac",
    }
    all_missing = {c for c in df.columns if df[c].isna().all()}

    def keep(excluded: set[str]) -> list[str]:
        return [c for c in df.columns if c not in excluded and c not in all_missing]

    return {
        "all_telemetry_features": keep(base_excluded | weak_or_constant),
        "no_time_features": keep(base_excluded | weak_or_constant | time_features),
        "style_core_no_time_context": keep(
            base_excluded | weak_or_constant | time_features | context_features
        ),
    }


def build_model(model_name: str, feature_cols: list[str]) -> Pipeline:
    # The concrete feature dtypes are only known at fit time, so the caller passes
    # a pre-split column list via DataFrame selection before building the pipeline.
    # This helper is kept for backwards compatibility with numeric-only calls.
    return build_model_for_frame(model_name, None, feature_cols)


def build_model_for_frame(model_name: str, X: pd.DataFrame | None, feature_cols: list[str]) -> Pipeline:
    if X is None:
        numeric_features = feature_cols
        categorical_features = []
    else:
        categorical_features = [c for c in feature_cols if X[c].dtype == "object"]
        numeric_features = [c for c in feature_cols if c not in categorical_features]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_features,
            )
        ],
        remainder="drop",
    )

    if model_name == "logistic_l2":
        model = LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="lbfgs",
            max_iter=2000,
            random_state=RANDOM_STATE,
        )
    elif model_name == "logistic_l1":
        model = LogisticRegression(
            penalty="l1",
            C=2.0,
            solver="saga",
            max_iter=2500,
            random_state=RANDOM_STATE,
        )
    elif model_name == "random_forest_regularized":
        model = RandomForestClassifier(
            n_estimators=160,
            max_depth=8,
            min_samples_leaf=5,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    elif model_name == "hist_gradient_boosting_regularized":
        model = HistGradientBoostingClassifier(
            learning_rate=0.04,
            max_iter=120,
            max_leaf_nodes=15,
            l2_regularization=0.2,
            random_state=RANDOM_STATE,
        )
    else:
        raise ValueError(model_name)

    return Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])


def make_direct_oof(
    df: pd.DataFrame,
    target_col: str,
    feature_cols: list[str],
    groups: np.ndarray,
    model_name: str,
) -> tuple[np.ndarray, LabelEncoder]:
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df[target_col].astype(str))
    min_count = pd.Series(y).value_counts().min()
    n_splits = min(N_SPLITS, int(min_count))
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = np.full(len(df), -1, dtype=int)
    X = df[feature_cols].copy()

    for train_idx, test_idx in cv.split(X, y, groups):
        model = build_model_for_frame(model_name, X.iloc[train_idx], feature_cols)
        model.fit(X.iloc[train_idx], y[train_idx])
        oof[test_idx] = model.predict(X.iloc[test_idx])
    return oof, label_encoder


def train_team_specific_driver_model(
    train_df: pd.DataFrame,
    team: str,
    feature_cols: list[str],
    model_name: str,
) -> tuple[Pipeline | None, LabelEncoder | None, str | None]:
    team_df = train_df[train_df["Team"] == team]
    drivers = sorted(team_df["Driver"].astype(str).unique())
    if len(drivers) == 0:
        return None, None, None
    if len(drivers) == 1:
        return None, None, drivers[0]

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(team_df["Driver"].astype(str))
    model = build_model_for_frame(model_name, team_df[feature_cols], feature_cols)
    model.fit(team_df[feature_cols], y)
    return model, label_encoder, None


def hierarchical_oof(
    df: pd.DataFrame,
    feature_cols: list[str],
    team_model_name: str,
    driver_model_name: str,
) -> tuple[np.ndarray, np.ndarray]:
    driver_encoder = LabelEncoder()
    team_encoder = LabelEncoder()
    y_driver = driver_encoder.fit_transform(df["Driver"].astype(str))
    y_team = team_encoder.fit_transform(df["Team"].astype(str))
    groups = df[session_column(df)].astype(str).to_numpy()

    min_count = pd.Series(y_team).value_counts().min()
    n_splits = min(N_SPLITS, int(min_count))
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    oof_team = np.full(len(df), -1, dtype=int)
    oof_driver = np.full(len(df), -1, dtype=int)
    X = df[feature_cols].copy()

    for train_idx, test_idx in cv.split(X, y_team, groups):
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        team_model = build_model_for_frame(team_model_name, train_df[feature_cols], feature_cols)
        team_model.fit(train_df[feature_cols], y_team[train_idx])
        pred_team_idx = team_model.predict(test_df[feature_cols])
        oof_team[test_idx] = pred_team_idx
        pred_team_names = team_encoder.inverse_transform(pred_team_idx)

        global_driver_model = build_model_for_frame(driver_model_name, train_df[feature_cols], feature_cols)
        global_driver_model.fit(train_df[feature_cols], y_driver[train_idx])

        local_models = {}
        for predicted_team in sorted(set(pred_team_names)):
            local_models[predicted_team] = train_team_specific_driver_model(
                train_df,
                predicted_team,
                feature_cols,
                driver_model_name,
            )

        for predicted_team in sorted(set(pred_team_names)):
            fold_positions = test_idx[pred_team_names == predicted_team]
            x_rows = df.iloc[fold_positions][feature_cols]
            model, local_encoder, constant_driver = local_models[predicted_team]
            if constant_driver is not None:
                predicted_drivers = [constant_driver] * len(fold_positions)
            elif model is not None and local_encoder is not None:
                local_pred = model.predict(x_rows)
                predicted_drivers = local_encoder.inverse_transform(local_pred)
            else:
                fallback_pred = global_driver_model.predict(x_rows)
                predicted_drivers = driver_encoder.inverse_transform(fallback_pred)
            oof_driver[fold_positions] = driver_encoder.transform(predicted_drivers)

    return oof_team, oof_driver


def dataset_rows(
    dataset_name: str,
    df: pd.DataFrame,
    feature_config: str,
    feature_cols: list[str],
    model_names: list[str],
) -> list[dict[str, object]]:
    group_col = session_column(df)
    groups = df[group_col].astype(str).to_numpy()

    rows = []
    for model_name in model_names:
        direct_driver_pred, driver_encoder = make_direct_oof(
            df, "Driver", feature_cols, groups, model_name
        )
        direct_team_pred, team_encoder = make_direct_oof(
            df, "Team", feature_cols, groups, model_name
        )
        y_driver = driver_encoder.transform(df["Driver"].astype(str))
        y_team = team_encoder.transform(df["Team"].astype(str))

        for result_name, target_name, y_true, y_pred, team_model_name, driver_model_name in [
            ("direct_driver", "Driver", y_driver, direct_driver_pred, None, model_name),
            ("direct_team", "Team", y_team, direct_team_pred, model_name, None),
        ]:
            rows.append(
                {
                    "dataset": dataset_name,
                    "feature_config": feature_config,
                    "result_name": result_name,
                    "target": target_name,
                    "team_model": team_model_name,
                    "driver_model": driver_model_name,
                    "n_samples": len(df),
                    "n_drivers": df["Driver"].nunique(),
                    "n_teams": df["Team"].nunique(),
                    "n_groups": df[group_col].nunique(),
                    "n_features": len(feature_cols),
                    "accuracy": accuracy_score(y_true, y_pred),
                    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
                    "macro_f1": f1_score(y_true, y_pred, average="macro"),
                }
            )

    hierarchical_pairs = [
        ("logistic_l2", "logistic_l2"),
        ("random_forest_regularized", "logistic_l2"),
        ("hist_gradient_boosting_regularized", "logistic_l2"),
        ("logistic_l2", "random_forest_regularized"),
        ("logistic_l2", "hist_gradient_boosting_regularized"),
    ]
    if feature_config != "style_core_no_time_context":
        return rows

    for team_model_name, driver_model_name in hierarchical_pairs:
            hierarchical_team_pred, hierarchical_driver_pred = hierarchical_oof(
                df, feature_cols, team_model_name, driver_model_name
            )
            driver_encoder = LabelEncoder().fit(df["Driver"].astype(str))
            team_encoder = LabelEncoder().fit(df["Team"].astype(str))
            y_driver = driver_encoder.transform(df["Driver"].astype(str))
            y_team = team_encoder.transform(df["Team"].astype(str))
            for result_name, target_name, y_true, y_pred in [
                ("hierarchical_team_stage", "Team", y_team, hierarchical_team_pred),
                ("hierarchical_driver_final", "Driver", y_driver, hierarchical_driver_pred),
            ]:
                rows.append(
                    {
                        "dataset": dataset_name,
                        "feature_config": feature_config,
                        "result_name": result_name,
                        "target": target_name,
                        "team_model": team_model_name,
                        "driver_model": driver_model_name,
                        "n_samples": len(df),
                        "n_drivers": df["Driver"].nunique(),
                        "n_teams": df["Team"].nunique(),
                        "n_groups": df[group_col].nunique(),
                        "n_features": len(feature_cols),
                        "accuracy": accuracy_score(y_true, y_pred),
                        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
                        "macro_f1": f1_score(y_true, y_pred, average="macro"),
                    }
                )
    return rows


def main() -> None:
    rows = []
    coverage_rows = []
    direct_model_names = [
        "logistic_l2",
        "random_forest_regularized",
        "hist_gradient_boosting_regularized",
    ]

    for dataset_name, config in DATASETS.items():
        path = config["path"]
        if not path.exists():
            continue
        df = pd.read_csv(path)
        df = df[df["Driver"].isin(config["drivers"])].copy()
        df = df.dropna(subset=["Driver", "Team"])
        for feature_config, cols in feature_sets(df).items():
            print(f"Running {dataset_name} / {feature_config}", flush=True)
            rows.extend(dataset_rows(dataset_name, df, feature_config, cols, direct_model_names))

        coverage = (
            df.groupby(["Driver", "Team"], dropna=False)
            .size()
            .reset_index(name="n_laps")
            .sort_values(["Driver", "n_laps"], ascending=[True, False])
        )
        coverage.insert(0, "dataset", dataset_name)
        coverage_rows.append(coverage)

    summary = pd.DataFrame(rows)
    best = (
        summary.sort_values(["dataset", "result_name", "macro_f1", "accuracy"], ascending=[True, True, False, False])
        .groupby(["dataset", "result_name"], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    coverage = pd.concat(coverage_rows, ignore_index=True)

    export_csv(summary, "hierarchical_team_driver_summary")
    export_csv(best, "hierarchical_team_driver_best_models")
    export_csv(coverage, "hierarchical_team_driver_coverage")

    print("Hierarchical team-driver analysis complete.")
    print(best.to_string(index=False))


# Notebook safety: odkomentuj, je?eli chcesz uruchomi? t? sekcj?.
# main()


## Model team-aware: prawdopodobie?stwa zespo?u jako cechy pomocnicze

?r?d?o: `08_team_aware_driver_models.py`

Je?eli chcesz uruchomi? oryginalny skrypt bez wchodzenia w definicje funkcji, u?yj:

```python
%run 08_team_aware_driver_models.py
```


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler


EXPORT_DIR = Path("exports")
RANDOM_STATE = 42
N_SPLITS = 5


DATASETS = {
    "short_qualifying_2023_2025_top5": {
        "path": EXPORT_DIR / "lap_features_model_min_40_laps.csv",
        "drivers": ["ALB", "SAI", "VER", "OCO", "LEC"],
        "sequence_summary": EXPORT_DIR / "sequence_architecture_summary.csv",
        "sequence_filter": {"model": ["hybrid_cnn_tabular", "cnn"]},
    },
    "long_qualifying_2018_2025_top5": {
        "path": EXPORT_DIR / "balanced_top5_lap_features_2018_2025_strict.csv",
        "drivers": ["SAI", "VER", "LEC", "OCO", "HAM"],
        "sequence_summary": EXPORT_DIR / "long_horizon_sequence_model_summary.csv",
        "sequence_filter": {"subset_name": ["balanced_top5"], "model": ["hybrid_cnn_tabular", "cnn"]},
    },
    "race_2025_clean_laps_top5": {
        "path": EXPORT_DIR / "race_2025_balanced_top5_lap_features.csv",
        "drivers": ["SAI", "VER", "LEC", "OCO", "HAM"],
        "sequence_summary": EXPORT_DIR / "race_2025_sequence_model_summary.csv",
        "sequence_filter": {"model": ["hybrid_cnn_tabular", "cnn"]},
    },
}


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def session_column(df: pd.DataFrame) -> str:
    if "session_key" in df.columns:
        return "session_key"
    return "round"


def style_feature_cols(df: pd.DataFrame) -> list[str]:
    excluded = {
        "lap_key",
        "Driver",
        "Team",
        "season",
        "round",
        "LapNumber",
        "event_name",
        "EventName",
        "session_name",
        "session_key",
        "TrackStatus",
        "lap_time_not_null",
        "dry_session",
        "accurate",
        "not_deleted",
        "not_fastf1_generated",
        "track_status_green",
        "not_pit_in_out",
        "within_2s_of_stint_median",
        "within_3s_of_stint_median",
        "LapTimeSeconds",
        "Sector1TimeSeconds",
        "Sector2TimeSeconds",
        "Sector3TimeSeconds",
        "sector_sum_seconds",
        "sector1_share",
        "sector2_share",
        "sector3_share",
        "stint_median_lap_seconds",
        "lap_time_delta_to_stint_median",
        "Compound",
        "TyreLife",
        "Stint",
        "SessionPart",
        "throttle_min",
        "brake_min",
        "brake_median",
        "brake_hard_frac",
        "drs_toggle_count",
        "brake_diff_mean",
        "drs_active_frac",
    }
    all_missing = {c for c in df.columns if df[c].isna().all()}
    return [c for c in df.columns if c not in excluded and c not in all_missing]


def build_logreg(feature_cols: list[str]) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                feature_cols,
            )
        ],
        remainder="drop",
    )
    model = LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=2000,
        random_state=RANDOM_STATE,
    )
    return Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])


def oof_team_probabilities(df: pd.DataFrame, feature_cols: list[str], groups: np.ndarray) -> tuple[np.ndarray, list[str]]:
    team_encoder = LabelEncoder()
    y_team = team_encoder.fit_transform(df["Team"].astype(str))
    classes = list(team_encoder.classes_)
    n_splits = min(N_SPLITS, int(pd.Series(y_team).value_counts().min()))
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    probs = np.zeros((len(df), len(classes)), dtype=float)
    X = df[feature_cols].copy()

    for train_idx, test_idx in cv.split(X, y_team, groups):
        model = build_logreg(feature_cols)
        model.fit(X.iloc[train_idx], y_team[train_idx])
        fold_probs = model.predict_proba(X.iloc[test_idx])
        for local_idx, class_idx in enumerate(model.named_steps["model"].classes_):
            probs[test_idx, class_idx] = fold_probs[:, local_idx]
    return probs, classes


def evaluate_driver_model(
    df: pd.DataFrame,
    feature_cols: list[str],
    groups: np.ndarray,
    extra_features: np.ndarray | None = None,
) -> dict[str, float]:
    driver_encoder = LabelEncoder()
    y_driver = driver_encoder.fit_transform(df["Driver"].astype(str))
    n_splits = min(N_SPLITS, int(pd.Series(y_driver).value_counts().min()))
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = np.full(len(df), -1, dtype=int)
    X_base = df[feature_cols].copy()

    if extra_features is not None:
        extra_cols = [f"team_prob_{i}" for i in range(extra_features.shape[1])]
        X_full = pd.concat(
            [X_base.reset_index(drop=True), pd.DataFrame(extra_features, columns=extra_cols)],
            axis=1,
        )
    else:
        X_full = X_base.reset_index(drop=True)

    full_cols = list(X_full.columns)
    for train_idx, test_idx in cv.split(X_full, y_driver, groups):
        model = build_logreg(full_cols)
        model.fit(X_full.iloc[train_idx], y_driver[train_idx])
        oof[test_idx] = model.predict(X_full.iloc[test_idx])

    return {
        "accuracy": accuracy_score(y_driver, oof),
        "balanced_accuracy": balanced_accuracy_score(y_driver, oof),
        "macro_f1": f1_score(y_driver, oof, average="macro"),
    }


def load_sequence_references(dataset_name: str, config: dict[str, object]) -> list[dict[str, object]]:
    path = config["sequence_summary"]
    if not path.exists():
        return []
    df = pd.read_csv(path)
    for col, allowed in config.get("sequence_filter", {}).items():
        if col in df.columns:
            df = df[df[col].isin(allowed)]
    rows = []
    for _, row in df.iterrows():
        rows.append(
            {
                "dataset": dataset_name,
                "model_family": "sequence_reference",
                "model_name": row["model"],
                "n_samples": int(row.get("n_samples", np.nan)),
                "accuracy": float(row["oof_accuracy"]),
                "balanced_accuracy": float(row["oof_balanced_accuracy"]),
                "macro_f1": float(row["oof_macro_f1"]),
                "notes": "existing sequence/hybrid result, not retrained in this script",
            }
        )
    return rows


def main() -> None:
    rows = []

    for dataset_name, config in DATASETS.items():
        df = pd.read_csv(config["path"])
        df = df[df["Driver"].isin(config["drivers"])].dropna(subset=["Driver", "Team"]).copy()
        feature_cols = style_feature_cols(df)
        groups = df[session_column(df)].astype(str).to_numpy()

        direct = evaluate_driver_model(df, feature_cols, groups)
        rows.append(
            {
                "dataset": dataset_name,
                "model_family": "tabular",
                "model_name": "direct_driver_logistic_l2",
                "n_samples": len(df),
                **direct,
                "notes": "style core features only",
            }
        )

        team_probs, team_classes = oof_team_probabilities(df, feature_cols, groups)
        team_aware = evaluate_driver_model(df, feature_cols, groups, extra_features=team_probs)
        rows.append(
            {
                "dataset": dataset_name,
                "model_family": "team_aware_tabular",
                "model_name": "driver_logistic_l2_plus_oof_team_probabilities",
                "n_samples": len(df),
                **team_aware,
                "notes": "style core features plus OOF team probability features: "
                + ", ".join(team_classes),
            }
        )

        rows.extend(load_sequence_references(dataset_name, config))

    summary = pd.DataFrame(rows)
    summary = summary.sort_values(["dataset", "macro_f1", "accuracy"], ascending=[True, False, False])
    best = summary.groupby("dataset", as_index=False).head(1).reset_index(drop=True)

    export_csv(summary, "team_aware_driver_model_comparison")
    export_csv(best, "team_aware_driver_model_best")

    print("Team-aware driver model comparison complete.")
    print(summary.to_string(index=False))


# Notebook safety: odkomentuj, je?eli chcesz uruchomi? t? sekcj?.
# main()
